# Greeley, CO — LVT Policy Analysis

Models a revenue-neutral split-rate LVT shift for parcels inside the **City of Greeley** corporate limits (Weld County, CO), following the canonical 7-section template in `.claude/skills/build-notebook.md`.

## Policy decisions (carried over from PR #8)

| Choice | Setting | Note |
|---|---|---|
| Geographic scope | City of Greeley (`LOCCITY = 'GREELEY'`) | Weld County FeatureServer attribute filter |
| Reform shape | Split-rate at 4:1 (land:improvement) | Lars's primary scenario |
| Land/improvement values | `LANDASD` / `IMPASD` (assessed values) | Weld County assessor schema |
| Excluded from solve | Exempt/Government, Agricultural, zoning H-A | Per Lars's `analysis_gdf` filter |
| Property classification | Two-tier: ACCTTYPE → PROPERTY_CATEGORY, refined to ANALYSIS_CATEGORY with zoning | Same as PR #8 |
| Current tax | **Proxy**: `TOTALASD / 1000` (1.0-mill placeholder) | Lars's choice — real Greeley city mill levy not modeled |

> **About the current-tax proxy.** Lars's PR #8 uses a placeholder 1.0-mill rate
> rather than Greeley's real city mill levy. This produces meaningful *relative*
> tax-shift results (which parcels gain vs lose, by what %) under a
> revenue-neutral solver, but **absolute dollar values are not real**. Replace
> the proxy with Greeley's actual city mill levy (or full stack levy) before
> publishing dollar figures.


## Section 1 — Imports and setup

In [ ]:
import sys
import json
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from shapely.geometry import Polygon

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# --- Constants ---
CITY_NAME = 'greeley'
STATE_FIPS = '08'
COUNTY_FIPS = '123'  # Weld County
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# --- Greeley-specific data sources (per Lars PR #8) ---
PARCELS_ENDPOINT = (
    'https://services.arcgis.com/ewjSqmSyHJnkfBLL/ArcGIS/rest/services/'
    'Parcels_open_data/FeatureServer/0/query'
)
ZONING_ENDPOINT = (
    'https://gis.greeleygov.com/arcgis_svr/rest/services/'
    'Data_services/Zoning/MapServer/2/query'
)
CITY_FILTER = "LOCCITY = 'GREELEY'"
PAGE_SIZE = 2000

ATTR_FIELDS = [
    'OBJECTID', 'PARCEL', 'ACCOUNTNO', 'ACCTTYPE', 'ACCOUNTTYP', 'NAME', 'SITUS', 'LOCCITY',
    'LANDACT', 'IMPACT', 'TOTALACT', 'LANDASD', 'IMPASD', 'TOTALASD',
    'LGLANDASD', 'LGIMPASD', 'TOTALLGASD', 'SCLANDASD', 'SCIMPASD', 'TOTALSCASD',
    'ASSRCODE', 'SALEP', 'SALEDT', 'GIS_Acres', 'latitude', 'longitude',
]

## Section 2 — Fetch parcel and zoning data

Two ArcGIS feature services:

1. **Weld County `Parcels_open_data`** (assessor attributes + geometry) — filtered to `LOCCITY = 'GREELEY'`
2. **Greeley city zoning** (`Data_services/Zoning/MapServer/2`) — for refined property classification

Both cached locally under `data/`.


In [ ]:
def fetch_arcgis_records(query_url, where, out_fields, page_size=2000, return_geometry=False):
    """Page through an ArcGIS query and return raw feature dicts."""
    session = requests.Session()
    count_resp = session.get(
        query_url,
        params={'f': 'json', 'where': where, 'returnCountOnly': 'true'},
        timeout=60,
    )
    count_resp.raise_for_status()
    total = int(count_resp.json().get('count', 0))
    print(f'Total matching: {total:,}')

    rows = []
    for offset in range(0, total, page_size):
        params = {
            'f': 'json',
            'where': where,
            'outFields': ','.join(out_fields),
            'returnGeometry': str(return_geometry).lower(),
            'resultOffset': offset,
            'resultRecordCount': page_size,
            'orderByFields': 'OBJECTID ASC',
        }
        if return_geometry:
            params['outSR'] = 4326
            params['geometryPrecision'] = 6
        resp = session.get(query_url, params=params, timeout=180)
        resp.raise_for_status()
        features = resp.json().get('features', [])
        if not features:
            break
        rows.extend(features)
    return rows


def esri_polygon_to_shapely(geom):
    rings = (geom or {}).get('rings')
    if not rings:
        return None
    return Polygon(rings[0], holes=rings[1:] if len(rings) > 1 else None)


def load_attrs():
    cache = DATA_DIR / 'greeley_attrs.parquet'
    if cache.exists():
        return pd.read_parquet(cache)
    features = fetch_arcgis_records(PARCELS_ENDPOINT, CITY_FILTER, ATTR_FIELDS, page_size=PAGE_SIZE)
    df = pd.DataFrame([f['attributes'] for f in features])
    df.to_parquet(cache, index=False)
    return df


def load_geometry():
    cache = DATA_DIR / 'greeley_geometry.parquet'
    if cache.exists():
        return gpd.read_parquet(cache)
    features = fetch_arcgis_records(
        PARCELS_ENDPOINT, CITY_FILTER, ['OBJECTID', 'PARCEL'],
        page_size=PAGE_SIZE, return_geometry=True,
    )
    rows = []
    for f in features:
        attrs = f.get('attributes', {})
        geom = esri_polygon_to_shapely(f.get('geometry') or {})
        if geom is not None:
            attrs['geometry'] = geom
            rows.append(attrs)
    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    gdf.to_parquet(cache, index=False)
    return gdf


def load_zoning():
    """Fetch Greeley zoning polygons. Server's class field is ZONING (not ZONING_CLASS)."""
    cache = DATA_DIR / 'greeley_zoning.parquet'
    if cache.exists():
        return gpd.read_parquet(cache)
    features = fetch_arcgis_records(
        ZONING_ENDPOINT, '1=1', ['OBJECTID', 'ZONING', 'ZONING_DESC'],
        page_size=PAGE_SIZE, return_geometry=True,
    )
    rows = []
    for f in features:
        attrs = f.get('attributes', {})
        geom = esri_polygon_to_shapely(f.get('geometry') or {})
        if geom is not None:
            attrs['geometry'] = geom
            rows.append(attrs)
    if not rows:
        # Empty / no features matched
        return gpd.GeoDataFrame(
            {'OBJECTID': [], 'ZONING': [], 'ZONING_DESC': [], 'geometry': []},
            geometry='geometry', crs='EPSG:4326',
        )
    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    gdf.to_parquet(cache, index=False)
    return gdf


t0 = time.time()
attrs_df = load_attrs()
geom_gdf = load_geometry()
print(f'Loaded {len(attrs_df):,} attribute rows, {len(geom_gdf):,} geometry rows in {time.time()-t0:.1f}s')

gdf = geom_gdf.merge(attrs_df, on=['OBJECTID', 'PARCEL'], how='inner')
gdf = gdf.drop_duplicates(subset=['OBJECTID']).sort_values('OBJECTID').reset_index(drop=True)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
print(f'Merged: {len(gdf):,} parcels')

# Coerce numerics
numeric_cols = ['LANDACT', 'IMPACT', 'TOTALACT', 'LANDASD', 'IMPASD', 'TOTALASD',
                'LGLANDASD', 'LGIMPASD', 'TOTALLGASD', 'SCLANDASD', 'SCIMPASD',
                'TOTALSCASD', 'SALEP', 'GIS_Acres']
for col in numeric_cols:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0)

In [ ]:
# Stamp zoning class onto parcels via spatial join (intersects)
zoning_gdf = load_zoning()
print(f'Zoning polygons: {len(zoning_gdf):,}')

gdf_3857 = gdf.to_crs(epsg=3857)
zoning_3857 = zoning_gdf.to_crs(epsg=3857)

# Use representative point of each parcel to avoid double-stamping when a parcel
# straddles a zoning boundary
gdf_3857['rep_point'] = gdf_3857.geometry.representative_point()
points_gdf = gpd.GeoDataFrame(
    gdf_3857[['OBJECTID']].copy(),
    geometry=gdf_3857['rep_point'],
    crs=gdf_3857.crs,
)
joined = gpd.sjoin(
    points_gdf,
    zoning_3857[['ZONING', 'geometry']],
    how='left', predicate='within',
)
zoning_map = joined.drop_duplicates(subset=['OBJECTID']).set_index('OBJECTID')['ZONING']
gdf['ZONING_CLASS'] = gdf['OBJECTID'].map(zoning_map).fillna('UNKNOWN')

print('Zoning class distribution (top 10):')
print(gdf['ZONING_CLASS'].value_counts().head(10))

## Section 3 — Property classification

Two-tier scheme from Lars PR #8:

1. **`PROPERTY_CATEGORY`** — assessor ACCTTYPE → base bucket (Residential, Commercial, Industrial, Vacant, Exempt/Government, Agricultural, Other)
2. **`ANALYSIS_CATEGORY`** — refined using zoning (Mixed Use breakouts, Residential density tiers)

Parcels in **Exempt/Government**, **Agricultural**, or zoning class **H-A** are excluded from the revenue-neutral solve.


In [ ]:
def map_property_category(accttype):
    v = str(accttype).upper()
    if 'VAC' in v or 'VACANT' in v:
        return 'Vacant / Undeveloped'
    if 'RES' in v or 'SINGLE' in v or 'CONDO' in v:
        return 'Residential'
    if 'COMM' in v or 'RETAIL' in v or 'OFFICE' in v:
        return 'Commercial'
    if 'IND' in v:
        return 'Industrial'
    if 'EXEM' in v or 'GOV' in v:
        return 'Exempt / Government'
    if 'AGRI' in v:
        return 'Agricultural'
    return 'Other'


def map_refined_category(base, zoning):
    z = str(zoning).upper()
    if base == 'Vacant / Undeveloped':
        return 'Vacant / Undeveloped'
    if 'MIX' in z or 'MU' in z:
        if z == 'MU-L':
            return 'Mixed Use - Low Density'
        if z == 'MU-H':
            return 'Mixed Use - High Density'
        return 'Mixed Use (Other Zoning)'
    if base == 'Residential':
        if z.startswith('R'):
            if 'R-H' in z:
                return 'Residential - High Density'
            if 'R-M' in z:
                return 'Residential - Medium Density'
            if 'R-MH' in z:
                return 'Residential - Mobile/Manufactured'
            if 'R-E' in z or 'R-L' in z:
                return 'Residential - Low Density'
        return 'Residential - Other'
    if base in ('Commercial', 'Industrial'):
        return base
    if z == 'P' or z.startswith('PUB'):
        return 'Public / Institutional'
    return base


gdf['PROPERTY_CATEGORY'] = gdf['ACCTTYPE'].apply(map_property_category)
gdf['ANALYSIS_CATEGORY'] = [
    map_refined_category(base, z)
    for base, z in zip(gdf['PROPERTY_CATEGORY'], gdf['ZONING_CLASS'])
]

print('PROPERTY_CATEGORY:')
print(gdf['PROPERTY_CATEGORY'].value_counts())
print()
print('ANALYSIS_CATEGORY (top 10):')
print(gdf['ANALYSIS_CATEGORY'].value_counts().head(10))

In [ ]:
# Exclusions: Exempt/Government, Agricultural, zoning H-A
gdf['excluded_from_solve'] = (
    gdf['PROPERTY_CATEGORY'].isin(['Exempt / Government', 'Agricultural'])
    | (gdf['ZONING_CLASS'] == 'H-A')
)
print(f'Excluded from solve: {gdf["excluded_from_solve"].sum():,} / {len(gdf):,}')
print(f'Solve population: {(~gdf["excluded_from_solve"]).sum():,}')

# Exemption flag (used by save_standard_export and downstream tools)
gdf['full_exmp'] = gdf['excluded_from_solve'].astype(int)

## Section 4 — Current tax (proxy)

Lars's PR #8 uses a placeholder 1.0-mill rate (`current_tax = TOTALASD / 1000`)
because Greeley's real city mill levy was not loaded into the source notebook.
Carrying that proxy forward unchanged. See top-of-notebook caveat — absolute
dollar values are not real, but the **relative** tax-shift output of the
revenue-neutral solver remains meaningful.


In [ ]:
PROXY_MILL_RATE = 1.0  # placeholder; replace with real Greeley city mill levy
gdf['millage_rate'] = PROXY_MILL_RATE
gdf['taxable_land_value'] = gdf['LANDASD'].clip(lower=0)
gdf['taxable_improvement_value'] = gdf['IMPASD'].clip(lower=0)
gdf['current_tax'] = (gdf['taxable_land_value'] + gdf['taxable_improvement_value']) * PROXY_MILL_RATE / 1000.0
gdf.loc[gdf['full_exmp'] == 1, 'current_tax'] = 0.0

current_revenue = float(gdf['current_tax'].sum())
print(f'Current revenue (proxy at {PROXY_MILL_RATE} mill): ${current_revenue:,.0f}')
print(f'Note: real Greeley city mill levy is ~11.27 mills (2024). Multiply outputs by ~11 for ballpark dollar figures.')

## Section 5 — Split-rate split (4:1)

Revenue-neutral solve at 4:1 (land:improvement). Excluded parcels stay at
`new_tax = current_tax = 0` and are recombined after the solve.


In [ ]:
solve_df = gdf[gdf['full_exmp'] == 0].copy()
print(f'Solving on {len(solve_df):,} parcels (excluded {len(gdf) - len(solve_df):,})')

land_millage, improvement_millage, new_revenue, solve_df = model_split_rate_tax(
    df=solve_df,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=float(solve_df['current_tax'].sum()),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine excluded parcels
excluded = gdf[gdf['full_exmp'] == 1].copy()
excluded['new_tax'] = 0.0
excluded['tax_change'] = 0.0
excluded['tax_change_pct'] = 0.0
gdf = pd.concat([solve_df, excluded]).sort_index()

print(f'\nLand millage:        {land_millage:.4f}')
print(f'Improvement millage: {improvement_millage:.4f}')
print(f'Revenue check: ${new_revenue:,.0f} vs current ${current_revenue:,.0f}')

In [ ]:
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f'{CITY_NAME} — 4:1 Split-Rate Tax Impact')

## Section 6 — Exploration chart

In [ ]:
import matplotlib
matplotlib.use('Agg')

fig, ax = plt.subplots(figsize=(10, 6))
summary = gdf.groupby('ANALYSIS_CATEGORY')['tax_change_pct'].median()
summary = summary[summary.index != 'Other'].sort_values()
summary.plot.barh(ax=ax, color=np.where(summary < 0, '#228B22', '#8B0000'))
ax.axvline(0, color='black', linewidth=1)
ax.set_title(f'{CITY_NAME} — Median Tax Change % by Analysis Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=120)
plt.close()
print(f'Saved category preview chart to {DATA_DIR / "category_preview.png"}')

## Section 7 — Census join + export

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False, census_categories=['Residential'])
print('Done.')